# Clustering Jerárquico Aglomerativo (HAC)
## Dataset: Hotel Bookings

El **Clustering Jerárquico Aglomerativo** (HAC, por sus siglas en inglés) es un algoritmo de aprendizaje no supervisado que construye una jerarquía de grupos de forma ascendente:
1. Cada observación comienza como su propio cluster.
2. En cada paso se fusionan los dos clusters más cercanos según el criterio de enlace elegido.
3. El proceso continúa hasta que todas las observaciones pertenecen a un único cluster.

La jerarquía resultante se visualiza con un **dendrograma**, que permite elegir el número óptimo de clusters cortando el árbol a la altura deseada.

### Métodos de enlace disponibles
| Método | Criterio de fusión |
|--------|--------------------|
| `ward` | Minimiza la varianza intra-cluster (recomendado) |
| `complete` | Distancia máxima entre pares de puntos |
| `average` | Distancia promedio entre pares de puntos |
| `single` | Distancia mínima entre pares de puntos |

## 1. Importación de librerías

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
import pckEDA as mf

%matplotlib inline
print('pckEDA cargado correctamente')
print('Clase HAC disponible:', mf.HAC)

## 2. Carga y exploración inicial del dataset

In [ ]:
datos = pd.read_csv('../Semana 8/hotel_bookings.csv', sep=',', decimal='.', index_col=0)
print(f'Dimensiones: {datos.shape}')
datos.head()

## 3. Preprocesamiento

### 3.1 Codificación de variables categóricas

In [ ]:
categoricas = [
    'hotel',
    'arrival_date_month',
    'assigned_room_type',
    'deposit_type',
    'customer_type',
    'reservation_status',
]

for col in categoricas:
    categorias = sorted(datos[col].dropna().unique())
    mapeo = {cat: i for i, cat in enumerate(categorias)}
    datos[col] = datos[col].map(mapeo)

print('Tipos de datos tras codificación:')
print(datos.dtypes)

### 3.2 Limpieza de nulos

In [ ]:
datos = datos.dropna()
print(f'Observaciones tras limpieza: {len(datos)}')

## 4. Creación del modelo HAC

Se utiliza el método de enlace **Ward** (minimiza varianza intra-cluster) con **4 clusters**.
Los datos se estandarizan automáticamente dentro de la clase.

In [ ]:
hac = mf.HAC(datos, n_clusters=4, metodo='ward')

print(f'Método de enlace : {hac.metodo}')
print(f'Número de clusters: {hac.n_clusters}')
print(f'Correlación cofenética: {hac.cophenet_corr:.4f}')
print()
print('Distribución de observaciones por cluster:')
print(pd.Series(hac.etiquetas).value_counts().sort_index())

> **Correlación cofenética**: valor entre 0 y 1 que mide qué tan bien la jerarquía preserva las distancias originales.  
> Un valor ≥ 0.70 se considera aceptable; valores ≥ 0.80 indican muy buena calidad.

## 5. Visualizaciones

### 5.1 Dendrograma

In [ ]:
plt.figure(figsize=(14, 6))
hac.plot_dendrograma(max_hojas=50, titulo='Dendrograma HAC — Hotel Bookings (Ward)')
plt.tight_layout()
plt.show()

### 5.2 Mapa de calor — Perfil de clusters

Muestra la media de cada variable por cluster, normalizada en desviaciones estándar respecto a la media global.

In [ ]:
plt.figure(figsize=(16, 5))
hac.plot_mapa_calor(titulo='Perfil de Clusters — Hotel Bookings')
plt.tight_layout()
plt.show()

### 5.3 Distribución de observaciones por cluster

In [ ]:
plt.figure(figsize=(8, 5))
hac.plot_distribucion(titulo='Distribución de Observaciones por Cluster')
plt.tight_layout()
plt.show()

### 5.4 Diagramas de dispersión por cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
hac.plot_dispersion('adr', 'lead_time', titulo='Tarifa Diaria vs Anticipación de Reserva')

plt.sca(axes[1])
hac.plot_dispersion('adr', 'stays_in_week_nights', titulo='Tarifa Diaria vs Noches entre Semana')

plt.tight_layout()
plt.show()

## 6. Resumen estadístico por cluster

In [ ]:
print('Media de cada variable por cluster:')
hac.resumen.style.background_gradient(cmap='Blues', axis=0).format('{:.2f}')

## 7. Comparación de métodos de enlace

Se comparan los cuatro métodos de enlace estándar para evaluar la calidad y estructura de los clusters.

In [ ]:
metodos = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, metodo in zip(axes.ravel(), metodos):
    hac_m = mf.HAC(datos, n_clusters=4, metodo=metodo)
    plt.sca(ax)
    hac_m.plot_dendrograma(
        max_hojas=30,
        titulo=f'Enlace: {metodo.capitalize()} — Cofenética: {hac_m.cophenet_corr:.3f}'
    )

plt.suptitle('Comparación de Métodos de Enlace — HAC', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Conclusiones

- El método **Ward** tiende a producir clusters de tamaño más equilibrado y obtiene la mejor correlación cofenética en la mayoría de los datasets tabulares.
- El **mapa de calor** revela los perfiles de comportamiento de cada cluster: grupos con alta tarifa (`adr`), grupos con largo tiempo de anticipación (`lead_time`), etc.
- Los **diagramas de dispersión** permiten detectar visualmente qué tan bien separados están los clusters en el espacio de variables originales.
- La **correlación cofenética** es el indicador de calidad principal: cuanto más cercana a 1, mejor preserva la jerarquía las distancias reales entre observaciones.